# 02 - Handoff Dataset for AI Engineer

Notebook ini membuat `modelling_dataset.csv` yang berisi fitur aktivitas harian dan target `stress_level`.

Dataset ini dibuat dari hasil inner join antara `daily_activities_clean` dan `stress_predictions_clean`, sehingga hanya baris aktivitas yang memiliki target valid yang masuk ke dataset modelling.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Clean Dataset

In [ ]:
# Membaca file CSV ke dalam dataframe.
daily_clean = pd.read_csv(PROCESSED_DIR / "daily_activities_clean.csv")
stress_predictions_clean = pd.read_csv(PROCESSED_DIR / "stress_predictions_clean.csv")

print("daily_clean:", daily_clean.shape)
print("stress_predictions_clean:", stress_predictions_clean.shape)

daily_clean: (26984, 18)
stress_predictions_clean: (26579, 7)


## 2. Inner Join

Inner join digunakan karena dataset modelling membutuhkan target. Aktivitas harian yang tidak memiliki prediction valid tidak digunakan pada dataset ini.

Dengan cara ini, `daily_activities_clean` tetap dapat disimpan sebagai data master, sementara `modelling_dataset.csv` hanya berisi data yang siap dipakai untuk training model.

In [ ]:
# menggabungkan dataset menggunakan key yang relevan untuk membentuk dataset analisis atau modelling.
# Menggabungkan dua dataset berdasarkan key yang relevan.
daily_joined = daily_clean.merge(
    stress_predictions_clean[["activity_id", "stress_score", "stress_level"]],
    left_on="id",
    right_on="activity_id",
    how="inner"
)

print("Rows before join - daily_clean:", len(daily_clean))
print("Rows before join - stress_predictions_clean:", len(stress_predictions_clean))
print("Rows after inner join:", len(daily_joined))

# Menampilkan beberapa baris awal untuk memahami bentuk data.
daily_joined.head()

Rows before join - daily_clean: 26984
Rows before join - stress_predictions_clean: 26579
Rows after inner join: 26579


,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at,activity_id,stress_score,stress_level
0,1,1,2026-01-01,7.23,1.93,5.78,3.81,13,160,4,7,5,4,6,2,5,2026-01-01 22:45:00,2026-01-01 23:39:00,1,49.60,Medium
1,2,1,2026-01-02,7.10,2.70,7.88,3.78,21,105,6,5,9,6,4,5,4,2026-01-02 19:44:00,2026-01-02 20:40:00,2,49.54,Medium
2,3,1,2026-01-03,6.56,3.24,8.65,4.78,23,123,6,6,5,5,8,2,6,2026-01-03 20:28:00,2026-01-03 20:33:00,3,50.63,Medium
3,4,1,2026-01-04,7.17,2.37,8.52,5.34,36,91,3,7,6,7,7,3,5,2026-01-04 23:55:00,2026-01-05 00:40:00,4,59.36,Medium
4,5,1,2026-01-05,7.60,5.20,7.43,4.82,29,103,6,6,7,9,2,5,6,2026-01-05 22:27:00,2026-01-05 23:02:00,5,56.98,Medium


## 3. Select Features and Target

In [ ]:
# menjalankan bagian kode pada tahap ini sesuai konteks notebook.
feature_columns = [
    "sleep_hours",
    "study_hours",
    "screen_time_hours",
    "social_media_hours",
    "physical_activity_minutes",
    "caffeine_intake_mg",
    "mood_score",
    "fatigue_level",
    "assignment_load",
    "deadline_pressure",
    "social_interaction_score",
    "financial_worry_score",
    "health_condition_score"
]

target_column = "stress_level"

modelling_dataset = daily_joined[feature_columns + [target_column]].copy()

# Menampilkan beberapa baris awal untuk memahami bentuk data.
modelling_dataset.head()

,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,stress_level
0,7.23,1.93,5.78,3.81,13,160,4,7,5,4,6,2,5,Medium
1,7.10,2.70,7.88,3.78,21,105,6,5,9,6,4,5,4,Medium
2,6.56,3.24,8.65,4.78,23,123,6,6,5,5,8,2,6,Medium
3,7.17,2.37,8.52,5.34,36,91,3,7,6,7,7,3,5,Medium
4,7.60,5.20,7.43,4.82,29,103,6,6,7,9,2,5,6,Medium


## Insight:

`stress_level` digunakan sebagai target klasifikasi. Kolom `stress_score` tidak dipakai sebagai fitur karena nilainya terlalu dekat dengan target dan dapat membuat model mempelajari output sistem, bukan pola dari aktivitas harian.

Dataset yang diserahkan ke AI Engineer berisi fitur yang relevan dengan aktivitas harian dan satu kolom target.

## 4. Final Check

In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Shape:", modelling_dataset.shape)

print("\nMissing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(modelling_dataset.isna().sum())

print("\nTarget distribution:")
# Menghitung distribusi nilai pada kolom kategorikal.
print(modelling_dataset[target_column].value_counts())

Shape: (26579, 14)

Missing value:
sleep_hours                  0
study_hours                  0
screen_time_hours            0
social_media_hours           0
physical_activity_minutes    0
caffeine_intake_mg           0
mood_score                   0
fatigue_level                0
assignment_load              0
deadline_pressure            0
social_interaction_score     0
financial_worry_score        0
health_condition_score       0
stress_level                 0
dtype: int64

Target distribution:
stress_level
Medium    23256
High       2583
Low         740
Name: count, dtype: int64


## 5. Save Modelling Dataset

In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
modelling_dataset.to_csv(PROCESSED_DIR / "modelling_dataset.csv", index=False)

report = "# Modelling Dataset Summary\n\n"
report += f"- daily_activities_clean rows: {len(daily_clean):,}\n"
report += f"- stress_predictions_clean rows: {len(stress_predictions_clean):,}\n"
report += f"- modelling_dataset rows: {len(modelling_dataset):,}\n\n"

report += "## Features\n\n"
for col in feature_columns:
    report += f"- {col}\n"

report += "\n## Target\n\n"
report += f"- {target_column}\n\n"

report += "## Important Note\n\n"
report += "`stress_score` tidak dimasukkan sebagai fitur untuk mencegah data leakage.\n"

# Menulis report dalam format markdown ke folder reports.
(REPORT_DIR / "modelling_dataset_summary.md").write_text(report, encoding="utf-8")

print("Saved:", PROCESSED_DIR / "modelling_dataset.csv")
print("Saved:", REPORT_DIR / "modelling_dataset_summary.md")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\modelling_dataset.csv
Saved: C:\Data Codingan\student_stress_data_science\outputs\reports\modelling_dataset_summary.md
